# Xenium single-sample analysis with Squidpy and NicheCompass

This Google Colab notebook is organized in the same workshop-style flow as the official 10x Genomics Xenium single-sample Python notebook, but adapted for your local Xenium output bundle and extended with **Squidpy neighborhood analysis** and **NicheCompass niche modeling**.

**Sample / output bundle**
- `output-XETG00081__0057246__1_1__20251128__031336`

**Workflow**
1. Install packages
2. Mount Google Drive
3. Read Xenium output bundle
4. Build AnnData / SpatialData representation
5. Basic QC and visualization
6. Clustering and marker exploration
7. Squidpy spatial neighborhood analysis
8. NicheCompass niche embedding and niche clustering
9. Save outputs

## 0. Runtime recommendation

- Runtime type: **GPU**
- This notebook is designed for **Google Colab**
- If NicheCompass installation fails, rerun the install cell once more

In [ ]:
# === 1. Install dependencies ===
!pip -q install "numpy<2" "anndata>=0.10,<0.11" "scanpy>=1.10,<1.11"                 "squidpy>=1.5,<1.7" "spatialdata>=0.2.5,<0.3" "spatialdata-io>=0.1.5"                 "matplotlib<3.9" "pandas<2.3" "scikit-learn<1.6" "h5py<3.13"                 "igraph" "leidenalg" "pyarrow" "shapely"

!pip -q install nichecompass==0.3.2

In [ ]:
# === 2. Imports ===
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import spatialdata as sd
import matplotlib.pyplot as plt

## 1. Mount Google Drive and define paths

Edit `PROJECT_ROOT` only if your folder is stored elsewhere.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === 3. Paths ===
PROJECT_ROOT = "/content/drive/MyDrive/Xenium"
XENIUM_DIR = os.path.join(PROJECT_ROOT, "output-XETG00081__0057246__1_1__20251128__031336")
OUT_DIR = os.path.join(PROJECT_ROOT, "colab_outputs_XETG00081")

os.makedirs(OUT_DIR, exist_ok=True)

print("XENIUM_DIR:", XENIUM_DIR)
print("OUT_DIR   :", OUT_DIR)
print("Exists?   :", os.path.exists(XENIUM_DIR))

In [ ]:
# Optional sanity check
for f in sorted(os.listdir(XENIUM_DIR))[:100]:
    print(f)

## 2. Read Xenium output bundle

This section follows the Python-oriented Xenium workflow style used in the official 10x notebook: load the output bundle, access the table object, and work from a cell-by-gene representation.

In [ ]:
# === 4. Read Xenium bundle with SpatialData ===
sdata = sd.read_zarr(XENIUM_DIR)

print(sdata)
print("Available tables:", list(sdata.tables.keys()) if hasattr(sdata, "tables") else "N/A")

In [ ]:
# === 5. Extract AnnData ===
# Most Xenium outputs expose the main expression table as 'table'
adata = sdata["table"].copy()

adata

In [ ]:
# === 6. Basic structure checks ===
print(adata.shape)
print(adata.obs.head())
print(adata.var.head())
print(adata.obsm.keys())

## 3. Prepare coordinates and counts layer

In [ ]:
# === 7. Make sure raw counts are available ===
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()

# === 8. Standardize spatial coordinates ===
# Xenium SpatialData objects commonly already have spatial coordinates,
# but we make them explicit for downstream tools.
if "spatial" not in adata.obsm.keys():
    possible_xy = [c for c in adata.obs.columns if c.lower() in ["x_centroid", "y_centroid", "x", "y"]]
    print("Possible coordinate columns:", possible_xy)

    if {"x_centroid", "y_centroid"}.issubset(set(adata.obs.columns)):
        adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy()
    elif {"x", "y"}.issubset(set(adata.obs.columns)):
        adata.obsm["spatial"] = adata.obs[["x", "y"]].to_numpy()
    else:
        raise ValueError("Could not find spatial coordinates in adata.obs / adata.obsm")

print(adata.obsm["spatial"][:5])

## 4. QC metrics and quick spatial visualization

In [ ]:
# === 9. QC metrics ===
adata.obs["n_counts"] = np.asarray(adata.layers["counts"].sum(axis=1)).ravel()
adata.obs["n_genes"] = np.asarray((adata.layers["counts"] > 0).sum(axis=1)).ravel()

adata.obs[["n_counts", "n_genes"]].describe()

In [ ]:
# === 10. QC plots ===
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(adata.obs["n_counts"], bins=50)
axes[0].set_title("n_counts per cell")
axes[1].hist(adata.obs["n_genes"], bins=50)
axes[1].set_title("n_genes per cell")
plt.tight_layout()
plt.show()

In [ ]:
# === 11. Optional filtering ===
adata = adata[(adata.obs["n_counts"] >= 20) & (adata.obs["n_genes"] >= 10)].copy()
adata

In [ ]:
# === 12. Quick spatial visualization ===
sc.pl.embedding(
    adata,
    basis="spatial",
    color="n_counts",
    size=6
)

## 5. Normalize, cluster, and inspect markers

This gives you the same practical single-sample analysis backbone most users expect after Xenium loading: normalization, PCA, graph clustering, UMAP, and marker inspection.

In [ ]:
# === 13. Normalization and feature selection ===
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)
adata = adata[:, adata.var["highly_variable"]].copy()

In [ ]:
# === 14. PCA / neighbors / clustering / UMAP ===
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=30)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.3, key_added="cluster")

sc.pl.umap(adata, color="cluster")

In [ ]:
# === 15. Top markers per cluster ===
sc.tl.rank_genes_groups(adata, groupby="cluster", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False)

In [ ]:
# === 16. Export marker table ===
markers = sc.get.rank_genes_groups_df(adata, group=None)
markers.to_csv(os.path.join(OUT_DIR, "XETG00081_markers.csv"), index=False)
markers.head(20)

## 6. Add manual cell-type annotations

Replace the mapping below after reviewing markers.

In [ ]:
# === 17. Manual annotation template ===
cluster_to_celltype = {
    "0": "Tumor",
    "1": "T_cell",
    "2": "Myeloid",
    "3": "Fibroblast",
}

adata.obs["cell_type"] = adata.obs["cluster"].astype(str).map(cluster_to_celltype)
adata.obs["cell_type"] = adata.obs["cell_type"].fillna("Unassigned")

sc.pl.umap(adata, color=["cluster", "cell_type"], wspace=0.4)

In [ ]:
# === 18. Spatial cell-type map ===
sc.pl.embedding(
    adata,
    basis="spatial",
    color="cell_type",
    size=6
)

## 7. Squidpy neighborhood analysis

This section extends the usual Xenium single-sample workshop by adding cell-cell neighborhood enrichment and co-occurrence analysis.

In [ ]:
# === 19. Spatial neighbor graph ===
sq.gr.spatial_neighbors(
    adata,
    coord_type="generic",
    spatial_key="spatial",
    n_neighs=6
)

# Symmetrize for downstream compatibility
adata.obsp["spatial_connectivities"] = adata.obsp["spatial_connectivities"].maximum(
    adata.obsp["spatial_connectivities"].T
)

In [ ]:
# === 20. Neighborhood enrichment ===
sq.gr.nhood_enrichment(adata, cluster_key="cell_type")
sq.pl.nhood_enrichment(adata, cluster_key="cell_type")

In [ ]:
# === 21. Co-occurrence ===
sq.gr.co_occurrence(adata, cluster_key="cell_type")
sq.pl.co_occurrence(adata, cluster_key="cell_type")

## 8. NicheCompass analysis

NicheCompass expects:
- a cell-level AnnData object
- spatial coordinates
- a precomputed spatial graph
- prior gene programs (GPs)

This notebook adds those pieces and trains a single-sample niche model.

In [ ]:
# === 22. NicheCompass imports ===
from nichecompass.models import NicheCompass
from nichecompass.utils import (
    add_gps_from_gp_dict_to_adata,
    extract_gp_dict_from_omnipath_lr_interactions,
    extract_gp_dict_from_nichenet_lrt_interactions,
    extract_gp_dict_from_mebocost_ms_interactions,
    filter_and_combine_gp_dict_gps_v2
)

In [ ]:
# === 23. Parameter keys ===
species = "human"   # change to "mouse" if needed

counts_key = "counts"
adj_key = "spatial_connectivities"

gp_names_key = "nichecompass_gp_names"
active_gp_names_key = "nichecompass_active_gp_names"
gp_targets_mask_key = "nichecompass_gp_targets"
gp_targets_categories_mask_key = "nichecompass_gp_targets_categories"
gp_sources_mask_key = "nichecompass_gp_sources"
gp_sources_categories_mask_key = "nichecompass_gp_sources_categories"
latent_key = "nichecompass_latent"
latent_cluster_key = "niche_cluster"

In [ ]:
# === 24. Build gene-program dictionary ===
omnipath_gp_dict = extract_gp_dict_from_omnipath_lr_interactions(
    species=species,
    load_from_disk=False,
    save_to_disk=False
)

nichenet_gp_dict = extract_gp_dict_from_nichenet_lrt_interactions(
    species=species,
    version="v2",
    keep_target_genes_ratio=1.0,
    max_n_target_genes_per_gp=250,
    load_from_disk=False,
    save_to_disk=False
)

mebocost_gp_dict = extract_gp_dict_from_mebocost_ms_interactions(
    species=species
)

combined_gp_dict = filter_and_combine_gp_dict_gps_v2(
    [omnipath_gp_dict, nichenet_gp_dict, mebocost_gp_dict],
    verbose=True
)

print("Number of combined gene programs:", len(combined_gp_dict))

In [ ]:
# === 25. Add gene programs to AnnData ===
add_gps_from_gp_dict_to_adata(
    gp_dict=combined_gp_dict,
    adata=adata,
    gp_targets_mask_key=gp_targets_mask_key,
    gp_targets_categories_mask_key=gp_targets_categories_mask_key,
    gp_sources_mask_key=gp_sources_mask_key,
    gp_sources_categories_mask_key=gp_sources_categories_mask_key,
    gp_names_key=gp_names_key,
    min_genes_per_gp=2,
    min_source_genes_per_gp=1,
    min_target_genes_per_gp=1
)

adata

In [ ]:
# === 26. Train NicheCompass model ===
model = NicheCompass(
    adata,
    counts_key=counts_key,
    adj_key=adj_key,
    gp_names_key=gp_names_key,
    active_gp_names_key=active_gp_names_key,
    gp_targets_mask_key=gp_targets_mask_key,
    gp_targets_categories_mask_key=gp_targets_categories_mask_key,
    gp_sources_mask_key=gp_sources_mask_key,
    gp_sources_categories_mask_key=gp_sources_categories_mask_key,
    latent_key=latent_key,
    conv_layer_encoder="gcnconv",
    active_gp_thresh_ratio=0.01
)

model.train(
    n_epochs=200,
    n_epochs_all_gps=25,
    lr=1e-3,
    lambda_edge_recon=500000.0,
    lambda_gene_expr_recon=300.0,
    lambda_l1_masked=0.0,
    edge_batch_size=1024,
    n_sampled_neighbors=4,
    use_cuda_if_available=True,
    verbose=False
)

In [ ]:
# === 27. Build niche embedding / cluster niches ===
sc.pp.neighbors(model.adata, use_rep=latent_key, key_added=latent_key)
sc.tl.umap(model.adata, neighbors_key=latent_key)
sc.tl.leiden(model.adata, neighbors_key=latent_key, resolution=0.4, key_added=latent_cluster_key)

model.adata.obs[latent_cluster_key].value_counts()

In [ ]:
# === 28. Visualize niches ===
sc.pl.umap(model.adata, color=["cell_type", latent_cluster_key], wspace=0.4)

sc.pl.embedding(
    model.adata,
    basis="spatial",
    color=["cell_type", latent_cluster_key],
    size=6
)

In [ ]:
# === 29. Niche composition table ===
ct = pd.crosstab(model.adata.obs[latent_cluster_key], model.adata.obs["cell_type"])
ct_frac = ct.div(ct.sum(axis=1), axis=0)

ct_frac

In [ ]:
# === 30. Niche composition heatmap ===
plt.figure(figsize=(8, 5))
plt.imshow(ct_frac.values, aspect="auto")
plt.xticks(range(ct_frac.shape[1]), ct_frac.columns, rotation=90)
plt.yticks(range(ct_frac.shape[0]), ct_frac.index)
plt.colorbar(label="fraction")
plt.title("Cell-type composition per niche")
plt.tight_layout()
plt.show()

## 9. Save outputs

In [ ]:
# === 31. Save outputs ===
adata.write_h5ad(os.path.join(OUT_DIR, "xenium_XETG00081_processed.h5ad"))
model.adata.write_h5ad(os.path.join(OUT_DIR, "xenium_XETG00081_nichecompass.h5ad"))

print("Saved:")
print(os.path.join(OUT_DIR, "xenium_XETG00081_processed.h5ad"))
print(os.path.join(OUT_DIR, "xenium_XETG00081_nichecompass.h5ad"))

## 10. Notes

- Update `cluster_to_celltype` after reviewing your marker genes.
- If your panel is narrow, NicheCompass can still cluster niches, but pathway-level interpretation may be limited.
- If installation fails on first run, rerun the install cell and restart the runtime.